In [1]:
import pandas as pd
import numpy as np
from scipy import stats
from statsmodels.formula.api import ols
from statsmodels.api import stats as sm
import statsmodels.stats.multicomp as mc

In [2]:
df=pd.read_excel(r"C:\Users\garvb\Downloads\Imp docs\Projects\AI Usage\AI On-Campus Research Survey (Responses).xlsx")
df=df.drop(columns="Timestamp",axis=1)
df.head()

,AI_Knowledge,Usage_AI,College_AI,Career_AI,ChatGPT,Stream
0,Knowledge5,Usage5,5,Career5,Yes,STEM
1,Knowledge4,Usage3,4,Career4,Yes,STEM
2,Knowledge3,Usage3,5,Career1,No,Business
3,Knowledge4,Usage3,5,Career2,Yes,Business
4,Knowledge5,Usage2,1,Career5,Yes,Humanities


In [3]:
cols=df.columns.tolist()
cols.remove("College_AI")
cols.remove("Usage_AI")
string="+".join(cols)

In [4]:
fit=ols(f"College_AI~{string}",data=df).fit()
two=pd.DataFrame(sm.anova_lm(fit,typ=2))
two=two[two["PR(>F)"]<0.05]
two

,sum_sq,df,F,PR(>F)
AI_Knowledge,22.486870,4.0,5.510319,0.000292
Career_AI,33.949097,4.0,8.319093,0.000003
Stream,21.174401,7.0,2.964973,0.005334


In [5]:
cols=two.index.tolist()
for i in cols:
    df[i]=df[i].astype(str)

In [6]:
pair=mc.MultiComparison(df["College_AI"],df["Stream"]+df["AI_Knowledge"]+df["Career_AI"])
test=pd.DataFrame(pair.tukeyhsd().summary()).tail(-1)
test[6]=test[6].astype(str)

In [7]:
test

,0,1,2,3,4,5,6
1,ArtsKnowledge1Career1,ArtsKnowledge2Career1,1.5,1.0,-3.4536,6.4536,False
2,ArtsKnowledge1Career1,ArtsKnowledge3Career1,0.5,1.0,-4.4536,5.4536,False
3,ArtsKnowledge1Career1,ArtsKnowledge3Career2,2.0,1.0,-2.6703,6.6703,False
4,ArtsKnowledge1Career1,ArtsKnowledge4Career1,1.0,1.0,-3.6703,5.6703,False
5,ArtsKnowledge1Career1,ArtsKnowledge4Career2,1.0,1.0,-4.7199,6.7199,False
...,...,...,...,...,...,...,...
2076,STEMKnowledge5Career5,TheologyKnowledge2Career1,-2.8333,0.915,-7.202,1.5353,False
2077,STEMKnowledge5Career5,TheologyKnowledge3Career1,-2.4333,0.0547,-4.8824,0.0158,False
2078,TheologyKnowledge1Career1,TheologyKnowledge2Career1,0.0,1.0,-5.7199,5.7199,False
2079,TheologyKnowledge1Career1,TheologyKnowledge3Career1,0.4,1.0,-4.0306,4.8306,False


In [8]:
pval=[]
for i in range(len(test)):
    df1=pd.DataFrame()
    df2=pd.DataFrame()
    
    df1=df[(df["Career_AI"]==(str(test.iloc[i][0])[-7:]))] 
    df1=df1[(df1["AI_Knowledge"]==(str(test.iloc[i][0])[-17:-7]))] 
    df1=df1[(df1["Stream"]==(str(test.iloc[i][0])[:-17]))]
    
    df2=df[(df["Career_AI"]==(str(test.iloc[i][1])[-7:]))] 
    df2=df2[(df2["AI_Knowledge"]==(str(test.iloc[i][1])[-17:-7]))] 
    df2=df2[(df2["Stream"]==(str(test.iloc[i][1])[:-17]))]
    
    
    if stats.ttest_ind(df1['College_AI'],df2['College_AI'],equal_var=True,alternative= 'greater')[1]<0.05 :
        pval.append(str(test.iloc[i][0]))
    else:
        pval.append(str(test.iloc[i][1]))
        
pval.remove("TheologyKnowledge2Career1")

c:\Users\garvb\AppData\Local\Programs\Python\Python311\Lib\site-packages\scipy\stats\_stats_py.py:7030: RuntimeWarning: invalid value encountered in scalar divide
  svar = ((n1 - 1) * v1 + (n2 - 1) * v2) / df
c:\Users\garvb\AppData\Local\Programs\Python\Python311\Lib\site-packages\scipy\stats\_axis_nan_policy.py:523: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, **kwds)
c:\Users\garvb\AppData\Local\Programs\Python\Python311\Lib\site-packages\scipy\stats\_stats_py.py:7030: RuntimeWarning: invalid value encountered in scalar divide
  svar = ((n1 - 1) * v1 + (n2 - 1) * v2) / df
c:\Users\garvb\AppData\Local\Programs\Python\Python311\Lib\site-packages\scipy\stats\_stats_py.py:7030: RuntimeWarning: invalid value encountered in scalar divide
  svar = ((n1 - 1) * v1 + (n2 - 1) * v2) / df
c:\Users\garvb\AppData\Local\Programs\Python\Pyth

In [9]:
df_final=pd.DataFrame()
df_final["Winners"]=pval
df_final["Winners"].value_counts()

STEMKnowledge5Career5        62
STEMKnowledge5Career3        60
TheologyKnowledge2Career1    59
STEMKnowledge4Career5        59
TheologyKnowledge1Career1    59
                             ..
BusinessKnowledge2Career1     6
ArtsKnowledge4Career1         6
BusinessKnowledge1Career1     4
ArtsKnowledge3Career1         3
ArtsKnowledge2Career1         3
Name: Winners, Length: 64, dtype: int64